## Positional Embeddings: Injecting Order into Transformers

### Why Positional Information Is Needed
Self-attention allows **all tokens to interact directly** with each other.  
While this is powerful, it introduces a problem:

- Unlike RNNs, Transformers **do not process tokens sequentially**
- Without extra information, the model has **no notion of order**
- The sentence “a cute teddy bear is reading” would look like an unordered set of tokens

To fix this, **positional information must be explicitly added**.

---

## Core Idea
Each token representation must encode:
- **What the token is**
- **Where the token appears in the sequence**

This is achieved using **positional embeddings**.

---

## Learned Positional Embeddings

### How It Works
- Each position in the sequence has its **own embedding**
  - Position 1 → embedding₁
  - Position 2 → embedding₂
  - …
- The positional embedding is **added element-wise** to the token embedding

**Example:**
- Token embedding for `"a"`  
- + positional embedding for position 1  
- → position-aware token representation

---

### How They Are Learned
- Positional embeddings are **trainable parameters**
- Learned via standard gradient descent during training
- Typically defined up to a **maximum sequence length** (e.g., 512)

---

### Limitations of Learned Positional Embeddings

1. **Sequence Length Dependency**
   - Only positions seen during training are learned
   - Tokens at unseen positions during inference have no learned embedding

2. **Training Data Bias**
   - If certain patterns always appear at specific positions in training data
   - The positional embeddings may encode those biases

---

### Why Use Them Anyway?
- Simple to implement
- Works well in practice
- Allows the model to **learn positional structure directly from data**

---

## Key Takeaway
Transformers require explicit position information because:
- Self-attention removes sequential ordering
- Positional embeddings reintroduce structure

**Learned positional embeddings** are one effective solution,  
but they come with generalization limits tied to training sequence length.

This motivates alternative approaches to encoding position.


## Fixed Positional Encodings: Sinusoidal Position Embeddings

### Two Approaches to Positional Information
So far, we have seen **learned positional embeddings**.  
There is a second approach where positional embeddings are **not learned**, but instead **predefined**.

In the original Transformer design, the authors chose a **sinusoidal formulation**.

---

## Core Idea
For each position `m`, create a vector of size `d_model` using **sine and cosine functions**.

Key constraints:
- Positional embedding dimension = token embedding dimension
- Embeddings are **added** to token embeddings
- No parameters are learned for positions

---

## Sinusoidal Formulation (High Level)
For each dimension `i`:
- One dimension uses a **sine**
- The next uses a **cosine**
- The frequency depends on the dimension index

Conceptually:
- Each dimension corresponds to a different wavelength
- Lower dimensions vary quickly (high frequency)
- Higher dimensions vary slowly (low frequency)

---

## Why Sine and Cosine?

### Desired Property
Tokens that are **closer in position** should have:
- More similar positional embeddings

Tokens that are **far apart** should:
- Be less similar

---

### Relative Position from Dot Products
When you take the **dot product** of two positional embeddings:
- The result becomes a function of **relative distance** between positions
- Maximum similarity occurs when positions are the same
- Similarity decreases as distance increases

This works because of trigonometric identities:
- Dot products of sine and cosine terms naturally encode relative offsets

This is important because:
- Similarity in embedding space is often measured using **dot products or cosine similarity**

---

## Frequency Intuition
- **Low-index dimensions** → high-frequency oscillations  
  - Capture fine-grained position changes
- **High-index dimensions** → low-frequency oscillations  
  - Capture coarse, long-range position information

Together, they allow the model to:
- Encode both local and global positional structure

---

## Advantages of Sinusoidal Positional Encodings

### 1. Generalization Beyond Training Length
- Can be computed for **any position**
- Works even for sequence lengths longer than those seen during training

### 2. No Learned Parameters
- No risk of overfitting to training-specific positional patterns
- Simpler and more stable

### 3. Comparable Performance
- Original results showed similar performance to learned positional embeddings

---

## Why This Matters
Transformers need positional information, but:
- Self-attention alone is order-agnostic
- Sinusoidal encodings reintroduce order in a principled way
- Relative distance becomes naturally encoded in the geometry of embeddings

---

## Looking Ahead
Modern models still preserve the **core intuition**:
- Nearby tokens should be more related than distant ones

However:
- Positional information is no longer injected exactly this way
- New approaches build on this idea while addressing its limitations

This sets the stage for **modern positional encoding methods**.


## Positional Information Inside the Attention Mechanism

### Why Input-Level Position Embeddings Are Not Enough
Earlier approaches added positional embeddings **to the input embeddings**.  
While this injects position information, it does so **indirectly**.

However, the place where **token similarity truly matters** is:
- Inside the **self-attention computation**
- Specifically inside the **softmax over query–key similarities**

This motivated new approaches that **modify attention directly**, rather than altering inputs.

---

## Where Similarity Is Computed
Self-attention computes relevance using:

$$
\text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

This softmax determines:
- Which tokens are more important
- How much each token contributes to another

To encode positional relationships more effectively:
- Positional information should influence **this similarity score itself**

---

## Injecting Position Information into Attention

### Core Idea
Instead of adding position embeddings to token embeddings:
- Add **position-dependent terms inside the attention score**
- Bias the attention mechanism so that:
  - Nearby tokens are favored
  - Distant tokens are penalized

This directly enforces positional intuition where it matters most.

---

## Relative Position Bias (Learned)

### Key Concept
- Compute the **relative distance** between token positions
- Group distances into buckets
- Learn a **bias value per bucket**
- Add this bias to the attention logits **before softmax**

### Properties
- Bias values are **learned**
- Allows the model to adapt positional behavior from data
- Softmax normalization still ensures valid probability distributions

This approach is used in models like **T5**.

---

## Deterministic Relative Bias: ALiBi

### Attention with Linear Bias (ALiBi)
Instead of learning biases:
- Use a **fixed, deterministic bias**
- Bias is a linear function of relative distance between tokens

### Intuition
- The farther two tokens are apart, the larger the penalty
- Encourages attention to focus on nearby tokens
- No extra learned parameters for position

This method generalizes well to **longer sequences than seen in training**.

---

## Why Softmax Is Not a Problem
Adding bias inside softmax is safe because:
- Softmax automatically normalizes scores
- Biases simply shift relative importance
- Final attention weights still sum to 1

---

## Key Takeaway
Modern models increasingly:
- Encode position **inside the attention computation**
- Use relative positional biases instead of absolute embeddings
- Directly influence similarity scores between tokens

While approaches differ (learned vs deterministic),  
the goal remains the same:
> **Make nearby tokens more likely to attend to each other than distant ones.**

This evolution leads to the positional encoding strategies used in today’s large language models.


## Rotary Position Embeddings (RoPE)

### Motivation
Earlier positional encoding methods had limitations:
- **Learned position embeddings** can overfit to training data and do not generalize beyond seen sequence lengths.
- **Relative position bias methods** (e.g., bucketed or linear bias) directly modify attention scores but can be either too data-dependent or too restrictive.

The goal remains the same:
> **Tokens that are closer in position should be more similar than tokens that are far apart — directly inside the attention mechanism.**

---

## Core Idea of RoPE
**Rotary Position Embeddings (RoPE)** encode position by **rotating the query and key vectors** by an angle that depends on their position.

Key points:
- Rotation is applied **inside the attention mechanism**
- Only **queries (Q)** and **keys (K)** are rotated
- Values (V) remain unchanged
- Positional information is injected **before computing attention scores**

---

## Intuition via Vector Rotation
Think of queries and keys as vectors in space:
- Each vector is rotated by an angle proportional to its position
- Rotation is done using a **rotation matrix**
- In 2D, rotation is achieved with a matrix containing sine and cosine terms

Effect:
- Rotating vectors changes their **relative orientation**
- Similarity between tokens becomes a function of **relative position**

---

## Why Rotation Works
In self-attention, similarity is computed using a **dot product**:

$$
Q \cdot K
$$

If:
- Query is rotated by angle proportional to position `m`
- Key is rotated by angle proportional to position `n`

Then:
- The dot product depends on **(n − m)**, the relative distance
- Absolute positions disappear
- Relative positions are preserved

This matches the desired behavior for attention.

---

## Extension to High Dimensions
Although rotation is easy to visualize in 2D:
- Transformer embeddings are **high-dimensional**
- RoPE applies rotation **pairwise across dimensions**
- Each pair of dimensions acts like a 2D rotation plane

The rotation angle:
- Is **fixed**
- Depends on embedding dimension index
- Uses the same sinusoidal frequency logic as earlier position encodings

---

## Advantages of RoPE

### 1. Relative Position Encoding
- Attention scores depend on **relative distance**
- No dependence on absolute position indices

### 2. No Learned Position Parameters
- Avoids overfitting to training-specific position patterns
- Generalizes naturally to longer sequences

### 3. Long-Range Decay
- Attention strength naturally **decays with distance**
- Distant tokens contribute less than nearby ones
- Decay is smooth (with mild oscillations due to trigonometry)

---

## Practical Impact
- RoPE is widely used in **modern large language models**
- Combines strengths of:
  - Sinusoidal encodings (generalization)
  - Relative position bias (direct attention control)
- Injects position exactly where similarity is computed

---

## Key Takeaway
RoPE encodes position by **rotating queries and keys**, making attention:
- Relative-position aware
- Length-generalizable
- Architecturally clean

This makes RoPE one of the most effective and widely adopted positional encoding methods in modern Transformer-based models.


## Normalization in Transformers: LayerNorm → Pre-Norm → RMSNorm

### Where Normalization Appears
In the Transformer architecture, each sublayer (Attention or FFN) is wrapped with an **Add & Norm** block.

Core pattern:
- Take the **input**
- Add it to the **output of the sublayer** (residual connection)
- Apply **normalization**

This design improves:
- Training stability
- Faster convergence
- Gradient flow in deep networks

---

## Why Normalization Is Needed
During training:
- Activations (intermediate vectors) can have **very large or very small values**
- Different layers may operate on very different value ranges
- This makes optimization harder

Normalization:
- Brings activations into a **controlled range**
- Makes learning more stable across layers

This phenomenon is often referred to as **internal covariate shift**.

---

## Layer Normalization (LayerNorm)

### How LayerNorm Works
Given an activation vector $( x $):
1. Compute the **mean** of its components
2. Compute the **standard deviation**
3. Normalize each component:
   $$
   \hat{x} = \frac{x - \mu}{\sigma}
   $$
4. Apply learned parameters:
   $$
   y = \gamma \hat{x} + \beta
   $$

Where:
- $( \gamma $) = learnable scale
- $( \beta $) = learnable shift

---

## Post-Norm vs Pre-Norm

### Post-Norm (Original Transformer)
Used in *Attention Is All You Need*:
$$
\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))
$$


### Pre-Norm (Modern Transformers)
Used in most modern models:

$$
\text{output} = x + \text{Sublayer}(\text{LayerNorm}(x))
$$


### Why Pre-Norm Is Preferred
- Better gradient flow in deep models
- More stable training
- Easier optimization when stacking many layers

---

## RMSNorm: A Simpler Alternative

### What Is RMSNorm?
**RMSNorm (Root Mean Square Normalization)** simplifies LayerNorm by:
- Removing mean subtraction
- Normalizing only by root mean square
- Learning **only one parameter (γ)**

Formula:
$$
\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2}} \cdot \gamma
$$

### Why RMSNorm Is Used Today
- Comparable convergence to LayerNorm
- Fewer parameters
- Faster computation
- Simpler implementation

As a result, **most modern LLMs use Pre-Norm + RMSNorm**.

---

## Why Not Batch Normalization?

### BatchNorm Normalizes:
- Across the **batch dimension**
- Uses statistics from multiple examples

### Problems in Transformers:
- Depends on batch size
- Different behavior during training vs inference
- Not well-suited for variable-length sequences

### Result:
Transformers favor **LayerNorm / RMSNorm**, which normalize **within each token representation**, independently of other samples.

---

## Key Takeaways
- Normalization stabilizes training and improves convergence
- Transformers evolved from **Post-Norm → Pre-Norm**
- **RMSNorm** is now widely used instead of LayerNorm
- BatchNorm is avoided due to batch dependence

Normalization is a critical but subtle component that enables Transformers to scale deeply and reliably.


## Efficient Attention Mechanisms: Scaling Beyond Quadratic Complexity

### The Core Challenge with Self-Attention
In standard self-attention:
- Every token attends to **every other token**
- For a sequence of length `n`, this results in **O(n²) complexity**
- As sequence length grows, memory and compute costs become prohibitive

This makes vanilla self-attention difficult to scale for long-context models.

---

## Approximating Full Attention

To reduce quadratic complexity **without sacrificing performance**, researchers explored constrained attention patterns.

One influential approach was introduced in **Longformer (2020)**.

---

## Sliding Window (Local) Attention

### Key Idea
Instead of attending to all tokens:
- Each token attends only to a **local neighborhood**
- Attention is restricted to a fixed-size window around the token

This reduces:
- Computation
- Memory usage

While preserving:
- Strong local context modeling

This pattern is commonly referred to as **sliding window attention**.

---

## How It Works in Practice
- Attention is **not computed for the full QKᵀ matrix**
- Implementations use optimized strategies (e.g. tiling, chunking)
- Only relevant local interactions are computed

This avoids explicitly forming large attention matrices.

---

## Local + Global Attention
Modern architectures often:
- Use **local attention** in some layers
- Use **global attention** in others
- **Interleave** these layers for better context propagation

There is no single optimal recipe — models experiment with different combinations.

---

## Window Size in Modern Models
- Illustrations often show small windows
- In practice, window sizes can be **thousands of tokens**
- Still far more efficient than full attention for very long sequences

---

## Analogy with Convolutions (Vision Insight)

Sliding window attention is conceptually similar to:
- **Convolutional receptive fields** in computer vision

Key intuition:
- A token may only attend locally in one layer
- But across multiple layers, information **propagates farther**
- Over depth, tokens indirectly interact with distant tokens

This mirrors how:
- Small convolution kernels
- Build large receptive fields across layers

---

## Practical Example
Some modern models (e.g. architectures with sliding window attention at every layer):
- Use local attention consistently
- Rely on **depth** to expand effective context
- Balance efficiency and expressiveness

---

## Key Takeaways
- Full self-attention scales as **O(n²)** and is expensive
- Sliding window attention restricts attention to local neighborhoods
- Efficient implementations avoid computing full attention matrices
- Local attention layers can be combined with global ones
- Deep stacking enables long-range interaction over time

Efficient attention is a key ingredient in making long-context Transformers practical.


## Attention Variants: Local Attention and Shared Projections

### Recap: Why Attention Variants Exist
Standard self-attention has **O(n²)** complexity because:
- Every token attends to every other token
- This becomes expensive for long sequences

To improve efficiency, modern Transformers introduce **orthogonal variations** to attention.

---

## Variation 1: Local (Sliding Window) Attention

### Idea
- Replace full `n × n` attention with **local attention**
- Each token attends only to a **fixed neighborhood** of nearby tokens

### Benefits
- Reduced computation and memory
- Scales better to long sequences
- Preserves strong local context

This approach is widely known as **sliding window attention** and is often combined with occasional global attention layers.

---

## Variation 2: Sharing Projection Matrices Across Heads

This variation focuses on **reducing memory and compute in multi-head attention**.

### Standard Multi-Head Attention
- Each head has its own:
  - Query projection
  - Key projection
  - Value projection
- Maximizes expressiveness but is expensive

---

## Motivation for Sharing Projections

### Key Observation
During decoding:
- The **same keys and values** are reused repeatedly
- Especially important in **autoregressive generation**

To support this efficiently, models use a **KV cache**:
- Stores previously computed keys and values
- Avoids recomputation at every decoding step

If each head has separate K and V projections:
- KV cache becomes very large

---

## Why Share K and V, but Not Q?

- **Queries** represent *how we ask the question*
  - Diversity across heads is valuable
- **Keys and values** represent *what is stored*
  - Reused across time steps
  - Memory-heavy during decoding

Sharing K and V:
- Reduces KV cache size
- Lowers memory footprint
- Improves inference efficiency

---

## Attention Sharing Configurations

### 1. Multi-Head Attention (MHA)
- Each head has its own Q, K, V projections
- Maximum flexibility
- Highest compute and memory cost

---

### 2. Grouped-Query Attention (GQA)
- Queries remain per-head
- Keys and values are shared across **groups of heads**
- Heads are divided into `g` groups

**Trade-off:**
- Good balance between performance and efficiency
- Common choice in modern LLMs

---

### 3. Multi-Query Attention (MQA)
- All heads share the **same** K and V projections
- Only queries differ per head

**Benefits:**
- Smallest KV cache
- Fastest decoding

**Cost:**
- Reduced expressiveness

---

## Where These Variants Are Used
- Most impactful in **decoder masked self-attention**
- Especially relevant for **decoder-only models**
- Can be applied to:
  - Self-attention
  - Cross-attention
  - Encoder–decoder attention

In practice:
- Modern LLMs are **decoder-only**
- Attention optimizations primarily target decoding efficiency

---

## Choosing the Right Variant
The choice depends on:
- Model size
- Sequence length
- Latency requirements
- Memory constraints
- Performance trade-offs

**Industry trend:**
- Many recent models favor **Grouped-Query Attention (GQA)**
- Provides strong performance with manageable resource usage

---

## Key Takeaways
- Attention variants improve scalability and efficiency
- Local attention reduces quadratic complexity
- Sharing K/V projections reduces memory and speeds up decoding
- MHA → GQA → MQA represents a spectrum of efficiency vs expressiveness
- GQA is a common modern default

These design choices are central to making large language models practical at scale.


## Transformer Model Families and Architectural Variants

### Revisiting the Encoder–Decoder Transformer
The original Transformer architecture (2017) consists of:
- An **encoder** to process the input sequence
- A **decoder** to generate the output sequence

This design was first introduced for **machine translation**, but later inspired multiple model families with different objectives and use cases.

---

## T5 Family: Text-to-Text Transformers

### What Is T5?
**T5** stands for **Text-to-Text Transfer Transformer**.

Key idea:
- Treat **every NLP task** as a text-to-text problem
- Both inputs and outputs are always text

---

### Variants of T5

- **T5 (base model)**  
  Original paper and formulation.

- **mT5 (multilingual T5)**  
  - Trained on multilingual data  
  - Larger, multilingual vocabulary

- **ByT5 (Byte-level T5)**  
  - No tokenization step
  - Operates directly on **bytes**
  - Vocabulary size = $(2^8 = 256$)
  - Trades longer sequences for simpler preprocessing

---

## Training Objective: Span Corruption (T5)

Unlike the original Transformer:
- Which uses **next-token prediction**

T5 uses **span corruption**:
- Random contiguous spans of tokens are removed from the input
- Each removed span is replaced with a **sentinel token**

### How It Works
- Encoder input: text with missing spans replaced by sentinel tokens
- Decoder output: sequential reconstruction of the missing spans
- Sentinel tokens delimit reconstructed spans

This forces the model to:
- Understand broader context
- Reconstruct missing information coherently

Training typically uses **teacher forcing**, where the full target sequence is provided during training.

---

## Encoder-Only Transformers

### Motivation
If the goal is **understanding**, not generation:
- The decoder is unnecessary

Encoder-only models:
- Produce strong contextual representations
- Are well-suited for **classification and extraction tasks**

---

### Common Encoder-Only Models
- **BERT**
- **DistilBERT** (lighter, faster version)
- **RoBERTa** (training and data improvements)

Typical use cases:
- Sentiment classification
- Token classification (NER)
- Sentence similarity
- Feature extraction

---

## Decoder-Only Transformers

### Architectural Shift
Modern large language models:
- **Remove the encoder entirely**
- Stack only **decoder blocks**

As a result:
- No cross-attention
- Only **masked self-attention + FFN**

---

### Why Decoder-Only Won
Key reasons:
- Simpler architecture
- Scales efficiently with compute and data
- Naturally aligned with **next-token prediction**
- Well-suited for open-ended generation

Next-token prediction proved to be:
- Easy to scale
- Highly generalizable
- Well-aligned with conversational and reasoning tasks

---

## Landscape Summary

| Architecture | Components | Strengths | Typical Tasks |
|-------------|----------|-----------|---------------|
| Encoder–Decoder | Encoder + Decoder | Sequence-to-sequence | Translation, structured generation |
| Encoder-Only | Encoder | Representation learning | Classification, extraction |
| Decoder-Only | Decoder | Scalable generation | LLMs, chat, reasoning |

---

## Key Takeaways
- Transformers evolved into **three major architectural families**
- T5 reframed NLP as text-to-text with span corruption
- Encoder-only models dominate understanding tasks
- Decoder-only models dominate modern LLMs
- Decoder-only architectures are the foundation of today’s large-scale generative models

These distinctions explain why modern LLMs look very different from the original Transformer — even though they share the same core ideas.


## Encoder-Only Transformers: BERT (Bidirectional Encoder Representations from Transformers)

---

## What does **BERT** mean?

**BERT** = **Bidirectional Encoder Representations from Transformers**

Each part of the acronym matters:

- **Encoder**  
  - Uses **only the encoder** stack of the Transformer  
  - Decoder is completely removed

- **Bidirectional**  
  - Each token can attend to **all tokens on the left and right**
  - No causal (future-masking) constraint
  - Contrast:
    - Encoder (BERT): full self-attention
    - Decoder (GPT): masked self-attention (causal)

- **Representations**  
  - Goal is to learn **contextual embeddings**
  - Optimized for understanding tasks, not generation

---

## Why Encoder-Only?

- No text generation
- Strong contextual representations
- Best suited for:
  - Classification
  - Sentence similarity
  - Token classification (NER)
  - QA-style understanding

---

## Historical Context

- **ELMo (2018)**  
  - Bidirectional LSTM-based embeddings  
  - Context-aware but:
    - Sequential (slow)
    - Hard to scale

- **BERT (2018)**  
  - Transformer-based
  - Fully parallelizable
  - Scales much better

---

## Special Tokens in BERT

### `[CLS]` (Classification Token)
- Added at the **beginning** of the sequence
- Final hidden state of `[CLS]` is used for:
  - Sentence-level classification
  - NSP task

### `[SEP]` (Separator Token)
- Separates two sentences
- Used for sentence-pair tasks

---

## Training Strategy: Two-Stage Training

### 1. Pre-training (Self-Supervised)
Learns general language representations using:
- **MLM** (Masked Language Modeling)
- **NSP** (Next Sentence Prediction)

### 2. Fine-tuning
- Attach a small task-specific head
- Train on labeled downstream task
- Requires relatively little labeled data

---

## Pre-training Objectives

### 1. Masked Language Model (MLM)

Procedure:
- Randomly select **15%** of tokens
  - **80%** → replaced with `[MASK]`
  - **10%** → replaced with a random token
  - **10%** → kept unchanged

Objective:
- Predict the **original token**
- Forces model to use **left + right context**

---

### 2. Next Sentence Prediction (NSP)

Procedure:
- Input: sentence pair (A, B)
- Labels:
  - **50%**: B is the true next sentence
  - **50%**: B is a random sentence

Objective:
- Binary classification using `[CLS]` embedding
- Learn sentence-level coherence

> Note: NSP was later questioned and removed in models like RoBERTa.

---

## Input Representation in BERT

Each token embedding is the **sum of three embeddings**:

1. **Token Embedding**
   - Learned lookup (WordPiece tokens)

2. **Position Embedding**
   - Absolute positional encoding
   - Added additively

3. **Segment Embedding**
   - Indicates sentence identity
   - Segment A → sentence 1
   - Segment B → sentence 2
   - Same segment embedding for all tokens in a sentence

---

## Tokenization: WordPiece

- Subword tokenizer
- Vocabulary size ≈ **30k**
- Balances:
  - Vocabulary size
  - OOV handling
- Typical vocab scale:
  - $(10^4$)–$(10^5$)
- Exception:
  - **ByT5** → byte-level (256 tokens)

---

## Encoder Architecture

- Same as Transformer encoder:
  - Multi-head self-attention
  - Feedforward network (FFN)
- Fully bidirectional self-attention
- Stack of identical encoder layers

---

## Notation Differences (BERT vs Transformer)

| Concept | Transformer | BERT |
|------|-----------|------|
| Sequence length | $(n$) | $(L$) |
| Embedding size | $(d_{model}$) | $(H$) |
| Attention heads | $(h$) | $(A$) |

(We usually keep Transformer notation for consistency.)

---

## Model Scale (Original BERT)

- Layers: **12**
- Hidden size: **768**
- Attention heads: **12**
- Parameters: ~**110M**

Variants:
- **BERT-base**
- **BERT-large**

---

## Pros and Cons of BERT

### Pros
- Learns strong contextual embeddings
- Uses large amounts of unlabeled data
- Excellent performance on understanding tasks
- Efficient fine-tuning

### Cons
- Cannot generate text
- Two-stage training (pre-train + fine-tune)
- NSP objective later shown to be unnecessary

---

## Key Takeaways

- BERT is **encoder-only** and **fully bidirectional**
- Uses **MLM + NSP** for pre-training
- `[CLS]` token enables sentence-level tasks
- Segment embeddings support sentence-pair tasks
- Foundation of modern NLP understanding models
- Inspired many improvements (RoBERTa, DistilBERT, etc.)


## BERT: Fine-Tuning Stage

---

## Goal of Fine-Tuning
- Reuse **pre-trained BERT weights**
- Adapt model to a **downstream task**
- Add a **task-specific head** (usually a linear layer)

---

## Fine-Tuning Strategies

### Option 1: Freeze Encoder
- Freeze all BERT encoder weights
- Train **only the classification head**
- Pros:
  - Faster
  - Less compute
- Cons:
  - Lower performance if task is very different

### Option 2: Full Fine-Tuning
- Train **entire model end-to-end**
- Encoder + task head updated together
- Pros:
  - Best performance
- Cons:
  - More compute
  - Risk of overfitting with small data

---

## Common Fine-Tuning Tasks

### 1. Sentence Classification (e.g., Sentiment Analysis)
- Use **[CLS] token embedding**
- Pipeline:
  - `[CLS]` embedding → Linear layer → Softmax
- Ignore other token embeddings

---

### 2. Token-Level Tasks (e.g., Question Answering)
- Use **all token embeddings**
- Two classifiers:
  - Start-position classifier
  - End-position classifier
- Predict answer span in the input text

---

## Example: BERT Input Pipeline

### Input Sentence
`this teddy bear is so cute`


### Tokenization (uncased)
- Lowercase text
- WordPiece tokenization

### Final Input Tokens
[CLS] this teddy bear is so cute [SEP] [PAD] ...


---

## Embedding Construction (per token)

Each token embedding =  
**Token Embedding**  
+ **Position Embedding**  
+ **Segment Embedding**

- Segment A: first sentence
- Segment B: second sentence
- Same segment embedding for all tokens in a segment

---

## Encoder Processing
- Tokens pass through stacked **Transformer encoders**
- Self-attention mixes context across all tokens
- Output embeddings are **contextual**

---

## Why Use Only `[CLS]` for Classification?
- `[CLS]` attends to **all tokens**
- Encodes full sentence meaning
- Convention chosen by BERT
- Other token embeddings are discarded **only for sentence-level tasks**

> For token-level tasks, **all embeddings are used**

---

## FFN (Classification Head)
- Maps embedding dimension → task output
- Typically:
  - Linear → (optional hidden layer) → Linear
- Trained during fine-tuning

---

## Query / Key / Value for `[CLS]`
- `[CLS]` treated like any other token
- Has its own Q, K, V
- Attends to all tokens
- Final embedding becomes sentence representation

---

## Strengths of BERT Fine-Tuning
- Contextual embeddings
- Flexible across tasks
- Strong performance with small labeled datasets
- Widely used in industry

---

## Limitations of BERT

1. **Context Length**
   - Fixed (typically 512 tokens)
   - Requires attention approximations for longer text

2. **Latency & Size**
   - ~110M parameters (BERT-base)
   - Slower inference

3. **Pre-training Complexity**
   - MLM + NSP
   - NSP later questioned by follow-up work

---

## Motivation for Later Models
- Reduce model size → **DistilBERT**
- Remove NSP → **RoBERTa**
- Improve efficiency & scalability

---

## Model Improvements After BERT

---

## Limitation: Cost & Model Size
- BERT is **large (~110M params)** and **slow**
- Need **smaller, faster models** with minimal performance loss

---

## Knowledge Distillation (Key Idea)

> *“The soft targets contain almost all the knowledge.”*  
— Hinton, Vinyals, Jeff Dean

### Core Concept
- Train a **student model** to mimic a **teacher model**
- Student learns from **soft probability distributions**, not hard labels

### Teacher–Student Setup
- **Teacher (T):** Large, well-trained model
- **Student (S):** Smaller, faster model

### Loss Function
- **KL Divergence** between teacher and student distributions
- If teacher output = hard labels → reduces to **cross-entropy**

---

## DistilBERT

### What They Changed
- Reduced number of layers by **~50%**
- Used **knowledge distillation** during training

### Result
- ~40% smaller
- ~60% faster
- **Minimal drop in performance**

### Key Insight
- Most knowledge lies in **output distributions**, not labels

---

## RoBERTa

### Questioned BERT Assumptions
- Is **NSP (Next Sentence Prediction)** really useful? → **No**

### Key Changes
- ❌ Removed NSP
- ✅ **Dynamic masking** (mask changes every epoch)
- ✅ Trained on **more data**
- ✅ Trained for **longer**

### Result
- Significant performance improvement
- Same architecture, better training strategy

---

## Big Takeaways
- **Distillation** → smaller & faster models
- **NSP not necessary**
- **Data scale + training strategy** matter as much as architecture

---
